# SDE simulation in a periodic potential

Simulate 50 independent trajectories of a one-dimensional stochastic differential equation with Diffrax, then plot the paths.

$$
dX_t = -Ak\sin(kX_t)\,dt + \sigma\,dW_t,
\qquad X_0 = 0.
$$

The drift comes from the periodic potential $V(x)=A[1-\cos(kx)]$. Near $x=0$, the drift is approximately $-Ak^2x$, giving a local Ornstein–Uhlenbeck approximation.


## 1. Imports


In [ ]:
import diffrax as dfx
import equinox as eqx
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import optax


## 2. Model parameters

$A$ controls the potential amplitude, $k$ sets its spatial period $2\pi/k$, and $\sigma$ is the constant diffusion coefficient. Each trajectory has its own Brownian motion.


In [ ]:
A = 1.0
k = 1.0
sigma = 1.1

n_trajectories = 50


## 3. Potential, drift, and diffusion

The deterministic drift points down the potential gradient:

$$
b(x)=-\frac{dV}{dx}=-Ak\sin(kx).
$$

Diffrax expects the drift and diffusion functions to accept `(t, x, args)`.


In [ ]:
def V(x):
    """Periodic potential."""
    return A * (1.0 - jnp.cos(k * x))


def drift(t, x, args):
    """Negative gradient of the potential."""
    return -A * k * jnp.sin(k * x)


def diffusion(t, x, args):
    """Constant diffusion coefficient."""
    return sigma


## 4. Time grid and initial condition

`dt` is the solver step size. `ts` contains the 15,000 times at which the solution is saved; its spacing is approximately 0.0333, so it differs from `dt`.


In [ ]:
t0 = 0.0
t1 = 500.0
dt = 0.01
x0 = 0.0

ts = jnp.linspace(t0, t1, 15_000)


## 5. Simulate one trajectory

Construct a Brownian path from a random key, combine the drift and diffusion terms, and integrate with the Euler solver. Return the state values at the saved times.


In [ ]:
def simulate_one(key):
    """Simulate one trajectory using the supplied random key."""
    brownian = dfx.VirtualBrownianTree(
        t0=t0,
        t1=t1,
        tol=1e-3,
        shape=(),
        key=key,
    )

    terms = dfx.MultiTerm(
        dfx.ODETerm(drift),
        dfx.ControlTerm(diffusion, brownian),
    )

    solution = dfx.diffeqsolve(
        terms,
        dfx.Euler(),
        t0=t0,
        t1=t1,
        dt0=dt,
        y0=x0,
        saveat=dfx.SaveAt(ts=ts),
        max_steps=100_000,
    )
    return solution.ys


## 6. Generate the ensemble

Split the fixed seed into one key per trajectory. `jax.vmap` applies the simulation to all keys. The resulting array has shape `(n_trajectories, len(ts))`, here `(50, 15000)`.


In [ ]:
key = jr.PRNGKey(42)
keys = jr.split(key, n_trajectories)

trajectories = jax.vmap(simulate_one)(keys)
print(f"Trajectories shape: {trajectories.shape}")


## 7. Plot the trajectories

Each line is an independent realization of the same SDE.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)

for trajectory in trajectories:
    ax.plot(ts, trajectory, color="black", linewidth=0.5, alpha=0.5)

ax.set(
    xlabel="t",
    ylabel="X(t)",
    title=f"{n_trajectories} SDE trajectories in a periodic potential",
)
ax.grid(alpha=0.2)
plt.show()


## 8. Learn the SDE by differentiating through Diffrax

Equinox defines the neural networks, and Optax supplies Adam. **Every training
loss calls Diffrax**, and `eqx.filter_value_and_grad` differentiates through the
numerical SDE solutions to update both networks.

We fit a conditional transition distribution: from an observed starting position,
simulate several possible future positions and compare their mean and variance
with the observed increment using a Gaussian score. This is a Monte Carlo
Gaussian approximation, not an exact transition likelihood. A single simulated
path has its own Brownian noise, so its pointwise error would be a poor way to
learn diffusion.

The drift receives `sin(x)` and `cos(x)`, which assumes the known spatial period
$2\pi$ ($k=1$). Diffusion receives `x` and is constrained to be positive. Both
networks have four hidden layers of width 64.


In [ ]:
class NeuralSDE(eqx.Module):
    drift_net: eqx.nn.MLP
    diffusion_net: eqx.nn.MLP

    def __init__(self, key):
        key_b, key_s = jr.split(key)
        self.drift_net = eqx.nn.MLP(
            in_size=2, out_size=1, width_size=64, depth=4,
            activation=jax.nn.tanh, key=key_b,
        )
        self.diffusion_net = eqx.nn.MLP(
            in_size=1, out_size=1, width_size=64, depth=4,
            activation=jax.nn.tanh, key=key_s,
        )

    def b(self, x):
        phase = jnp.stack([jnp.sin(x), jnp.cos(x)])
        return self.drift_net(phase)[0]

    def s(self, x):
        raw = self.diffusion_net(jnp.atleast_1d(x))[0]
        return jax.nn.softplus(raw) + 1e-3


model = NeuralSDE(jr.PRNGKey(100))


### Prepare observed transitions

Use the first 40 trajectories for training and reserve the remaining 10 for
validation. Training transitions span 10 saved intervals (about 0.333 time
units). This provides a stronger drift signal than a single saved interval;
Diffrax models the evolution of the drift and diffusion throughout that window.
It also reduces the relative effect of interpolation in the original saved data.

Keep the original single-interval arrays `x`, `dx`, and `delta` for the direct
estimates later in the notebook.


In [ ]:
train_paths = trajectories[:40]
validation_paths = trajectories[40:]
transition_stride = 10

# Single-interval data used by the later plots and direct estimates.
x = train_paths[:, :-1].reshape(-1)
dx = jnp.diff(train_paths, axis=1).reshape(-1)
delta = jnp.broadcast_to(
    jnp.diff(ts), train_paths[:, :-1].shape
).reshape(-1)


def make_transitions(paths):
    sampled = paths[:, ::transition_stride]
    elapsed = jnp.diff(ts[::transition_stride])
    return (
        sampled[:, :-1].reshape(-1),
        jnp.diff(sampled, axis=1).reshape(-1),
        jnp.broadcast_to(elapsed, sampled[:, :-1].shape).reshape(-1),
    )


train_x, train_dx, train_delta = make_transitions(train_paths)
validation_x, validation_dx, validation_delta = make_transitions(validation_paths)
print(f"Training transitions: {train_x.size:,}")
print(f"Validation transitions: {validation_x.size:,}")


### Simulate a short transition with trainable coefficients

For an interval of length $h$, rescale time to $u\in[0,1]$:

$$
dY_u = h\,b_\theta(Y_u)\,du + \sqrt{h}\,s_\theta(Y_u)\,dB_u.
$$

Thus every solve uses the same time domain, with the observed elapsed time passed
in `args` alongside the network parameters. Euler integrates the Itô SDE using
16 internal steps, and `RecursiveCheckpointAdjoint` backpropagates through them.

For each starting position, generate 16 independent Brownian paths and solve
with both $B$ and $-B$, giving 32 simulated increments. These antithetic pairs
reduce noise in the estimated mean. The variance is calculated across all
increments; paired paths are not independent observations.

Diffrax documents this differentiation method in its [adjoint guide](https://docs.kidger.site/diffrax/api/adjoints/#diffrax.RecursiveCheckpointAdjoint).


In [ ]:
training_substeps = 16
brownian_pairs = 16


def transition_drift(t, y, args):
    model, elapsed, sign = args
    return elapsed * model.b(y)


def transition_diffusion(t, y, args):
    model, elapsed, sign = args
    return sign * jnp.sqrt(elapsed) * model.s(y)


def simulate_transition(model, start, elapsed, key, sign):
    brownian = dfx.VirtualBrownianTree(
        t0=0.0, t1=1.0, tol=1e-3, shape=(), key=key,
    )
    terms = dfx.MultiTerm(
        dfx.ODETerm(transition_drift),
        dfx.ControlTerm(transition_diffusion, brownian),
    )
    solution = dfx.diffeqsolve(
        terms, dfx.Euler(),
        t0=0.0, t1=1.0, dt0=1.0 / training_substeps,
        y0=start, args=(model, elapsed, sign),
        saveat=dfx.SaveAt(t1=True),
        adjoint=dfx.RecursiveCheckpointAdjoint(),
        max_steps=training_substeps + 2,
    )
    return solution.ys[0] - start


def transition_moments(model, start, elapsed, key):
    keys = jr.split(key, brownian_pairs)
    positive = jax.vmap(
        lambda k: simulate_transition(model, start, elapsed, k, 1.0)
    )(keys)
    negative = jax.vmap(
        lambda k: simulate_transition(model, start, elapsed, k, -1.0)
    )(keys)
    increments = jnp.concatenate([positive, negative])
    mean = jnp.mean(increments)
    variance = jnp.mean((increments - mean)**2)
    return mean, jnp.maximum(variance, 1e-6)


### Score the simulated transition distribution

Let $\widehat\mu_\theta(x,h)$ and $\widehat v_\theta(x,h)$ be the mean and variance
of the Diffrax-generated increments. For an observed increment $\Delta x$, use

$$
\ell(\theta)=\frac12\left[
\frac{(\Delta x-\widehat\mu_\theta)^2}{\widehat v_\theta}
+\log\widehat v_\theta\right].
$$

The squared error rewards the right mean; the variance scaling and logarithm
encourage the right fluctuation size. Gradients flow through both simulated
moments, through the solver, and into the drift and diffusion networks. Resample
the Brownian keys at each training step. Finite Monte Carlo samples, numerical
steps, and the Gaussian approximation all affect the estimates.


In [ ]:
def loss_fn(model, x_batch, dx_batch, delta_batch, simulation_key):
    keys = jr.split(simulation_key, x_batch.size)
    means, variances = jax.vmap(
        lambda start, elapsed, key: transition_moments(model, start, elapsed, key)
    )(x_batch, delta_batch, keys)
    return jnp.mean(
        0.5 * ((dx_batch - means)**2 / variances + jnp.log(variances))
    )


### Update the model with Optax Adam

Equinox differentiates only the array parameters and keeps activation functions
static during compilation. Optax updates the parameters using Adam; gradients
still pass through the Diffrax solves in `loss_fn`.

References: [Equinox transformations](https://docs.kidger.site/equinox/api/transformations/) and [Optax Adam](https://optax.readthedocs.io/en/latest/api/generated/optax.adam.html).


In [ ]:
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(eqx.filter(model, eqx.is_inexact_array))


@eqx.filter_jit
def train_step(model, opt_state, xb, dxb, dtb, simulation_key):
    loss, grads = eqx.filter_value_and_grad(loss_fn)(
        model, xb, dxb, dtb, simulation_key
    )
    updates, opt_state = optimizer.update(
        grads, opt_state, eqx.filter(model, eqx.is_inexact_array)
    )
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss


### Train and track validation loss

Use fixed validation observations and Brownian keys for comparable scores.
Validation data never enter parameter updates. Record every training loss and
check validation every 100 updates, including the initial and final model.

Training includes many SDE solves; the first update also compiles the forward
and backward computations.


In [ ]:
batch_size = 16
training_steps = 1500
validation_every = 100
validation_size = 64
validation_indices = jr.randint(
    jr.PRNGKey(102), (validation_size,), 0, validation_x.size
)
validation_batch = (
    validation_x[validation_indices],
    validation_dx[validation_indices],
    validation_delta[validation_indices],
)
validation_key = jr.PRNGKey(103)
evaluate_loss = eqx.filter_jit(loss_fn)
initial_validation_loss = float(evaluate_loss(model, *validation_batch, validation_key))
print(f"Initial validation loss: {initial_validation_loss:.4f}")

key = jr.PRNGKey(101)
loss_history = []
validation_steps = [0]
validation_history = [initial_validation_loss]
for step in range(training_steps):
    key, batch_key, simulation_key = jr.split(key, 3)
    indices = jr.randint(batch_key, (batch_size,), 0, train_x.size)
    model, opt_state, loss = train_step(
        model, opt_state,
        train_x[indices], train_dx[indices], train_delta[indices],
        simulation_key,
    )
    loss_history.append(float(loss))
    completed_steps = step + 1
    if completed_steps % validation_every == 0 or completed_steps == training_steps:
        validation_loss = float(evaluate_loss(model, *validation_batch, validation_key))
        validation_steps.append(completed_steps)
        validation_history.append(validation_loss)
        print(
            f"Step {completed_steps}: training = {float(loss):.4f}, "
            f"validation = {validation_loss:.4f}"
        )

final_validation_loss = validation_history[-1]


## 9. Training progress

Mini-batches and simulated Brownian paths make the training loss noisy. The
moving average shows its trend; validation uses fixed observations and keys.
Lower scores are better, and Gaussian scores can be negative.


In [ ]:
training_losses = np.asarray(loss_history)
training_updates = np.arange(1, len(training_losses) + 1)
smoothing_window = min(50, len(training_losses))
smoothed_loss = np.convolve(
    training_losses, np.ones(smoothing_window) / smoothing_window, mode="valid"
)

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.plot(training_updates, training_losses, color="tab:blue", alpha=0.2,
        linewidth=0.7, label="Training (each batch)")
ax.plot(training_updates[smoothing_window - 1:], smoothed_loss,
        color="tab:blue", label=f"Training ({smoothing_window}-step average)")
ax.plot(validation_steps, validation_history, "o-", color="tab:orange",
        label="Validation (fixed sample)")
ax.set(xlabel="Optimizer updates", ylabel="Gaussian transition score",
       title="Training and validation loss")
ax.grid(alpha=0.2)
ax.legend()
plt.show()


## 10. Learned drift and diffusion

Compare the learned functions with the generating coefficients over the central
98% of training positions. The histogram shows where observations provide
support; estimates in rarely visited regions are less constrained.


In [ ]:
grid = jnp.linspace(jnp.percentile(x, 1), jnp.percentile(x, 99), 300)
learned_b = jax.vmap(model.b)(grid)
learned_s = jax.vmap(model.s)(grid)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
axes[0].plot(grid, learned_b, label="Learned")
axes[0].plot(grid, -A * k * jnp.sin(k * grid), "--", label="True")
axes[0].set(xlabel="x", ylabel="b(x)", title="Drift")
axes[1].plot(grid, learned_s, label="Learned")
axes[1].axhline(sigma, linestyle="--", color="black", label="True")
axes[1].set(xlabel="x", ylabel="s(x)", title="Diffusion")
axes[2].hist(np.asarray(x), bins=80, density=True, color="tab:gray", alpha=0.7)
axes[2].set(xlabel="x", ylabel="Density", title="Training positions")
axes[2].set_xlim(float(grid[0]), float(grid[-1]))
for ax in axes[:2]:
    ax.legend()
    ax.grid(alpha=0.2)
plt.show()


## 11. Compare parameter estimates once

Assuming the known $k=1$, project the learned drift onto $-k\sin(kx)$ to obtain an
effective $A$, and average the learned diffusion over observed positions for an
effective $\sigma$. These summarize networks that can represent more general
functions than the original model.

The direct baseline fits the small-interval Gaussian likelihood using the same
40 training trajectories. Its drift fit weights squared residuals by inverse
elapsed time. Both approaches are approximate; numerical saving and finite
observation intervals can affect the estimates.


In [ ]:
k_known = 1.0
sample_x = x[::100]
basis_sample = -k_known * jnp.sin(k_known * sample_x)
b_from_net = jax.vmap(model.b)(sample_x)
s_from_net = jax.vmap(model.s)(sample_x)
A_neural = jnp.sum(basis_sample * b_from_net) / jnp.sum(basis_sample**2)
sigma_neural = jnp.mean(s_from_net)

basis = -k_known * jnp.sin(k_known * x)
predictor = basis * delta
A_direct = jnp.sum(predictor * dx / delta) / jnp.sum(predictor**2 / delta)
residual = dx - A_direct * predictor
sigma_direct = jnp.sqrt(jnp.mean(residual**2 / delta))

print(f"{'Method':<20} {'A':>10} {'sigma':>10}")
print("-" * 42)
for name, a_value, s_value in (
    ("Generating values", A, sigma),
    ("Direct estimate", A_direct, sigma_direct),
    ("Neural summary", A_neural, sigma_neural),
):
    print(f"{name:<20} {float(a_value):>10.4f} {float(s_value):>10.4f}")


## 12. Simulate a path from the fitted model

Plot one learned path alongside a held-out observed path. The paths use different
Brownian noise, so their pointwise agreement is not a fitting criterion. Use the
loss and coefficient plots above to assess learning.


In [ ]:
def learned_drift(t, x, args):
    return args.b(x)


def learned_diffusion(t, x, args):
    return args.s(x)


brownian = dfx.VirtualBrownianTree(
    t0=t0, t1=t1, tol=1e-3, shape=(), key=jr.PRNGKey(202)
)
terms = dfx.MultiTerm(
    dfx.ODETerm(learned_drift),
    dfx.ControlTerm(learned_diffusion, brownian),
)
learned_path = dfx.diffeqsolve(
    terms, dfx.Euler(),
    t0=t0, t1=t1, dt0=dt, y0=x0, args=model,
    saveat=dfx.SaveAt(ts=ts),
    max_steps=100_000,
).ys

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, sharey=True,
                         constrained_layout=True)
axes[0].plot(ts, validation_paths[0], color="tab:gray", linewidth=0.8)
axes[0].set(ylabel="X(t)", title="Held-out observed path")
axes[1].plot(ts, learned_path, color="tab:blue", linewidth=0.8)
axes[1].set(xlabel="t", ylabel="X(t)", title="Path from the learned SDE")
for ax in axes:
    ax.grid(alpha=0.2)
plt.show()
